In [12]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv(r"C:\Users\glotzie\Downloads\archive\electric_vehicles_spec_2025.csv.csv")

feature_columns = [
    "battery_capacity_kWh",
    "top_speed_kmh",
    "torque_nm",
    "efficiency_wh_per_km",
    "acceleration_0_100_s",
    "fast_charging_power_kw_dc",
    "towing_capacity_kg",
    "cargo_volume_l",
    "length_mm",
    "width_mm",
    "height_mm"
]

# Convert columns to numeric, replacing non-numeric values with NaN
for col in feature_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Also convert the target variable to numeric
df["range_km"] = pd.to_numeric(df["range_km"], errors='coerce')

# Drop rows with missing values in feature columns or target variable
df_clean = df[feature_columns + ["range_km"]].dropna()

# Extract features and target from cleaned data
X = df_clean[feature_columns].values
y = df_clean["range_km"].values

np.random.seed(42)

indices = np.random.permutation(len(X))

train_size = int(0.8*len(X))

train_idx = indices[:train_size]
test_idx = indices[train_size:]

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

# Standardize features
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train = (X_train-mean)/std
X_test = (X_test-mean)/std

# Add bias term (intercept)
ones = np.ones((X_train.shape[0],1))
X_train = np.hstack((ones,X_train))

ones = np.ones((X_test.shape[0],1))
X_test = np.hstack((ones,X_test))

theta = np.zeros(X_train.shape[1])

def compute_cost(X,y,theta):
    m=len(y)
    predictions=X@theta
    error=predictions-y
    cost=(error@error)/(2*m)
    return cost

def compute_gradient(X,y,theta):
    m=len(y)
    error=(X@theta)-y
    gradient=(X.T@error)/m
    return gradient

def gradient_descent(X,y,theta,alpha,iterations):
    history=[]
    for i in range(iterations):
        gradient=compute_gradient(X,y,theta)
        theta=theta-alpha*gradient
        history.append(compute_cost(X,y,theta))
    return theta,history

theta=np.zeros(X_train.shape[1])

theta,cost_history=gradient_descent(
    X_train,
    y_train,
    theta,
    alpha=0.01,
    iterations=5000
)

y_pred=X_test@theta

# Model evaluation 
rmse=np.sqrt(np.mean((y_test-y_pred)**2))
mae=np.mean(np.abs(y_test-y_pred))
ss_res=np.sum((y_test-y_pred)**2)
ss_tot=np.sum((y_test-y_test.mean())**2)
r2=1-ss_res/ss_tot

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R²: {r2}")

RMSE: 23.584350712053357
MAE: 18.54897835218389
R²: 0.9481288528557958
